# Riesgo de baja actividad en CREA — análisis y modelado

Datos de estudiantes de Plan Ceibal, extractos anuales 2019–2025.

Este notebook sigue el enfoque estándar de un proyecto de machine learning, en ocho pasos:

1. Importar las librerías necesarias
2. Leer el dataset y obtener una visión general
3. Análisis exploratorio de datos — a. Univariado  b. Bivariado
4. Preprocesamiento de datos
5. Definir la métrica de desempeño y construir los modelos
6. Verificación de supuestos
7. Comparar modelos y determinar el mejor
8. Observaciones e insights de negocio

**Problema.** Predecir si un estudiante tendrá actividad total igual a cero en CREA
durante el **siguiente** año lectivo, a partir de sus datos del año en curso. El uso
previsto es análisis de alcance y priorización de apoyo educativo; **no** debe utilizarse
para decisiones automatizadas sobre estudiantes individuales.

**Cómo ejecutar.** Ejecute las celdas en orden. Los dos datasets intermedios (~500MB cada
uno) se reutilizan si ya existen en `DataSets/`; para regenerarlos desde los siete
extractos anuales ponga `REBUILD_COMBINED = True` y `REBUILD_CLEAN = True` en la celda de
configuración del paso 1.

**Por qué el paso 3 va antes del 4.** El análisis exploratorio se hace sobre los datos
*sin limpiar*, a propósito: así los problemas de calidad (dos variantes del placeholder
`"Sin Dato"`, tres notaciones distintas de `grado`, pares `id_persona`+`año_lectivo`
duplicados) quedan a la vista y justifican cada regla del paso 4, en lugar de
desaparecer antes de mostrarse.

## 1. Importar las librerías necesarias

In [ ]:
import csv
import gc
import json
import sys
import time
from itertools import islice
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import dump
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier

# Resolve the repository layout before importing the local module. Jupyter already puts
# the notebook's own directory on sys.path; doing it explicitly also covers a kernel
# started at the repository root.
SCRIPTS_DIR = Path.cwd().resolve()
if not (SCRIPTS_DIR / "train_engagement_risk.py").exists():
    SCRIPTS_DIR = SCRIPTS_DIR / "Scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# The modeling helpers are imported from the training script rather than copied, so the
# script and this notebook cannot drift apart: same cohort construction, same feature
# selection, same chronological cross-validation.
from train_engagement_risk import (
    build_cohort,
    build_pipeline,
    choose_activity_column,
    normalize_keys,
    select_model_features,
    select_threshold,
    time_series_cross_validate,
)

%matplotlib inline
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

BASE_DIR = SCRIPTS_DIR.parent
DATASETS_DIR = BASE_DIR / "DataSets"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
RANDOM_STATE = 42

# Reuse the ~500MB intermediates when they exist. Rebuilding means re-reading the seven
# yearly extracts (~3 min), cleaning 4.5M rows (~5 min) and two ~500MB writes.
REBUILD_COMBINED = False
REBUILD_CLEAN = False

# 4.5M rows make every distribution plot illegible and slow. Aggregates and tables are
# computed on the full frame; only the plots draw from this reproducible sample.
PLOT_SAMPLE = 300_000

ACTIVITY_COLUMNS = [
    "cantidad_de_días_ingreso_a_crea",
    "cantidad_de_entregas_de_tareas",
    "cantidad_de_comentarios_posteados",
    "cantidad_de_acciones_totales",
    "cantidad_de_días_de_ingreso_a_matific",
    "cantidad_de_episodios_finalizados_en_matific",
    "cantidad_de_días_de_ingreso_a_pam",
    "cantidad_de_actividades_finalizadas_en_pam",
    "cantidad_de_días_de_ingreso_a_biblioteca",
    "cantidad_de_préstamos_en_biblioteca",
]
CATEGORICAL_COLUMNS = [
    "sexo",
    "rol",
    "departamento",
    "subsistema",
    "ciclo",
    "grado",
    "zona",
    "contexto",
]

print("Directorio base:", BASE_DIR)
print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

## 2. Leer el dataset y obtener una visión general

Los siete extractos anuales viven en `DataSets/datos_estudiantes_AAAA.csv`. Cada archivo
puede traer un delimitador distinto y los nombres de columna cambian entre años, así que
la lectura pasa primero por una capa de estandarización.

### 2a. Funciones auxiliares de lectura

In [ ]:
# Only the per-year extracts are inputs. Globbing "*.csv" would also pick up this
# notebook's own outputs and fold them back into the next run.
YEARLY_PATTERN = "datos_estudiantes_[0-9][0-9][0-9][0-9].csv"
COMBINED_NAME = "datos_estudiantes_total.csv"
CLEAN_NAME = "datos_estudiantes_total_clean.csv"

# The 2025 extract renames three CREA measures with an "en CREA" suffix while earlier
# years use the unsuffixed names. Mapping them onto one name keeps the combined frame at
# one column per measure; without this, each name is missing for the years that use the
# other spelling, and those gaps look like real data once anything fills them in.
COLUMN_ALIASES = {
    "cantidad_de_entregas_de_tareas_en_crea": "cantidad_de_entregas_de_tareas",
    "cantidad_de_comentarios_posteados_en_crea": "cantidad_de_comentarios_posteados",
    "cantidad_de_acciones_totales_en_crea": "cantidad_de_acciones_totales",
}


def get_csv_files():
    csv_files = sorted(DATASETS_DIR.glob(YEARLY_PATTERN))

    if not csv_files:
        print(f"No yearly CSV files found in {DATASETS_DIR}")
        return []

    return csv_files


def detect_delimiter(path: Path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        sample = file.read(4096)
        file.seek(0)

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;")
        return dialect.delimiter
    except csv.Error:
        return ";" if ";" in sample else ","


def standardize_column_name(column_name):
    standardized = str(column_name).strip().lower().replace(" ", "_")
    return COLUMN_ALIASES.get(standardized, standardized)


def get_standardized_columns(csv_file: Path):
    delimiter = detect_delimiter(csv_file)

    with csv_file.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, delimiter=delimiter)
        header = next(reader, None)

    if header is None:
        return []

    standardized = [standardize_column_name(column) for column in header]
    return standardized


def load_csv_as_dataframe(csv_file: Path):
    delimiter = detect_delimiter(csv_file)
    df = pd.read_csv(csv_file, sep=delimiter, encoding="utf-8-sig")
    df.columns = [standardize_column_name(col) for col in df.columns]
    return df

### 2b. Vista previa de los archivos crudos

Encabezado y primeras filas de cada extracto, para ver los nombres originales y el
delimitador detectado antes de cargar nada en memoria.

In [ ]:
def print_first_rows(csv_file: Path, rows_to_show: int = 5):
    """Show the delimiter, the standardized header and the first rows of one extract.

    Reads only the rows displayed: ``list(reader)`` would materialize all ~650k rows of
    a 70MB file just to print five of them.
    """
    delimiter = detect_delimiter(csv_file)

    with csv_file.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, delimiter=delimiter)
        rows = list(islice(reader, rows_to_show))

    print(f"\n=== {csv_file.name} ===")
    if not rows:
        print("File is empty.")
        return

    print("Delimiter:", repr(delimiter))
    print("Columns:", get_standardized_columns(csv_file))
    for row in rows:
        print(row)


for csv_file in get_csv_files():
    print_first_rows(csv_file)

### 2c. Combinar los extractos anuales

`report_coverage` muestra qué fracción de cada columna está observada en cada año. Es la
forma de ver el *schema drift*: un `0.0` significa que ese año no reporta la columna, y
esa distinción tiene que sobrevivir hasta el modelo.

In [ ]:
def report_coverage(frame: pd.DataFrame):
    """Show the observed fraction of each column per year to expose schema drift."""
    coverage = (
        frame.notna().groupby(frame["año_lectivo"]).mean().round(3).sort_index().T
    )
    print("\nObserved fraction by year (0.0 means the year lacks the column):")
    print(coverage.to_string())
    return coverage


def create_combined_dataset(output_name: str = COMBINED_NAME):
    csv_files = get_csv_files()
    if not csv_files:
        return None

    dataframes = [load_csv_as_dataframe(csv_file) for csv_file in csv_files]
    combined = pd.concat(dataframes, ignore_index=True)
    combined = combined.drop_duplicates()

    output_path = DATASETS_DIR / output_name
    combined.to_csv(output_path, index=False)

    print(f"Combined dataset saved to: {output_path}")
    print(f"Rows: {len(combined):,}")
    print(f"Columns: {list(combined.columns)}")
    return combined


def load_combined_dataset(path: Path):
    """Read the combined extract with the categoricals declared.

    Left to itself pandas keeps these eight columns as Python strings and the frame costs
    ~2.6GB; as ``category`` the same data takes ~0.5GB. That matters because the cleaning
    step in section 4 needs a second copy alongside this one.
    """
    frame = pd.read_csv(
        path, low_memory=False, dtype={column: "category" for column in CATEGORICAL_COLUMNS}
    )
    print(f"Loaded {path.name}: {len(frame):,} rows x {frame.shape[1]} columns")
    return frame


combined_path = DATASETS_DIR / COMBINED_NAME
if REBUILD_COMBINED or not combined_path.exists():
    combined = create_combined_dataset()
else:
    print(f"Reusing existing {COMBINED_NAME} (set REBUILD_COMBINED = True to rebuild).")
    combined = load_combined_dataset(combined_path)

if combined is None:
    raise FileNotFoundError(
        f"No hay extractos anuales en {DATASETS_DIR} que coincidan con {YEARLY_PATTERN}, "
        f"ni un {COMBINED_NAME} previo para reutilizar."
    )

coverage = report_coverage(combined)

### 2d. Visión general del dataset

Dimensiones, tipos, faltantes, cardinalidad y duplicados. Dos lecturas para retener de
acá: la cobertura por año de la celda anterior y el conteo de pares
`id_persona`+`año_lectivo` duplicados, que `drop_duplicates()` no elimina porque esas
filas difieren en alguna columna.

In [ ]:
print("Dimensiones:", combined.shape)
print(f"Memoria: {combined.memory_usage(deep=True).sum() / 1e9:.2f} GB")
display(combined.head())

In [ ]:
combined.info(show_counts=True)

overview = pd.DataFrame(
    {
        "dtype": combined.dtypes.astype(str),
        "no_nulos": combined.notna().sum(),
        "faltantes_%": (combined.isna().mean() * 100).round(2),
        "valores_distintos": combined.nunique(dropna=True),
    }
)
display(overview)

In [ ]:
display(combined[ACTIVITY_COLUMNS].describe().T)
display(combined[CATEGORICAL_COLUMNS].astype("object").describe().T)

In [ ]:
print("Filas duplicadas exactas:", int(combined.duplicated().sum()))
print(
    "Pares id_persona+año_lectivo duplicados:",
    int(combined.duplicated(["id_persona", "año_lectivo"]).sum()),
)
print("Estudiantes únicos:", combined["id_persona"].nunique())
display(
    combined.groupby("año_lectivo")["id_persona"].agg(filas="size", estudiantes_únicos="nunique")
)

## 3. Análisis exploratorio de datos

Sobre el dataset combinado **antes** de limpiarlo. Los aspectos que interesan son tres:
la forma de las métricas de actividad (todas con exceso de ceros y cola larga), la
estructura de los faltantes (que no es aleatoria) y los defectos de codificación que el
paso 4 tiene que corregir.

### 3a. Análisis univariado

In [ ]:
def summarize_activity(frame: pd.DataFrame) -> pd.DataFrame:
    """Per-metric shape: coverage, zero inflation and the upper tail.

    Zeros and missing values are reported separately on purpose. A zero is a measured
    "did not use it"; a blank is a year that never reported the metric at all. Collapsing
    the two would invent activity data for the years with no PAM columns.
    """
    rows = {}
    for column in ACTIVITY_COLUMNS:
        if column not in frame.columns:
            continue
        values = pd.to_numeric(frame[column], errors="coerce")
        observed = values.notna()
        rows[column] = {
            "observado_%": round(observed.mean() * 100, 2),
            "ceros_%_del_observado": round((values[observed] == 0).mean() * 100, 2),
            "media": round(float(values.mean()), 2),
            "mediana": float(values.median()),
            "p95": float(values.quantile(0.95)),
            "p99.5": float(values.quantile(0.995)),
            "máximo": float(values.max()),
        }
    return pd.DataFrame(rows).T


activity_summary = summarize_activity(combined)
display(activity_summary)

In [ ]:
plot_sample = combined.sample(n=min(PLOT_SAMPLE, len(combined)), random_state=RANDOM_STATE)
sample_activity = plot_sample[ACTIVITY_COLUMNS].apply(pd.to_numeric, errors="coerce")

fig, axes = plt.subplots(4, 3, figsize=(16, 14))
for ax, column in zip(axes.ravel(), ACTIVITY_COLUMNS):
    values = sample_activity[column].dropna()
    # log1p, not the raw value: every metric piles up at zero and then reaches into the
    # tens of thousands, so a linear axis draws one bar at zero and nothing else.
    ax.hist(np.log1p(values), bins=50, color="#4C72B0")
    ax.set_title(column.replace("cantidad_de_", ""), fontsize=9)
    ax.set_xlabel("log1p(valor)")
    ax.set_ylabel("frecuencia")
for ax in axes.ravel()[len(ACTIVITY_COLUMNS):]:
    ax.axis("off")
fig.suptitle("Distribución de las métricas de actividad (escala log1p)", y=1.0)
fig.tight_layout()
plt.show()

In [ ]:
melted = (
    np.log1p(sample_activity)
    .melt(var_name="métrica", value_name="log1p(valor)")
    .dropna()
)
melted["métrica"] = melted["métrica"].str.replace("cantidad_de_", "", regex=False)

plt.figure(figsize=(13, 6))
sns.boxplot(data=melted, x="log1p(valor)", y="métrica", fliersize=1)
plt.title("Dispersión y valores extremos por métrica (escala log1p)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 20))
for ax, column in zip(axes.ravel(), CATEGORICAL_COLUMNS):
    counts = combined[column].value_counts(dropna=False).head(20)
    labels = [str(index) for index in counts.index]
    ax.barh(labels[::-1], counts.to_numpy()[::-1], color="#55A868")
    ax.set_title(f"{column} — {combined[column].nunique()} valores distintos", fontsize=10)
    ax.set_xlabel("filas")
fig.suptitle("Distribución de las variables categóricas (sin limpiar, top 20)", y=1.0)
fig.tight_layout()
plt.show()

In [ ]:
# Three concrete defects, visible in the value counts and each one a rule in section 4.
print("1) Dos variantes del mismo placeholder en 'zona':")
print(combined["zona"].value_counts(dropna=False).to_string())
print("\n2) 'contexto' con su propio placeholder ('sin clasificar'):")
print(combined["contexto"].value_counts(dropna=False).head(10).to_string())
print(f"\n3) 'grado': {combined['grado'].nunique()} valores para ~6 niveles, en tres notaciones")
print(combined["grado"].value_counts(dropna=False).head(20).to_string())
print("\n'rol' es constante y por lo tanto no aporta señal:")
print(combined["rol"].value_counts(dropna=False).to_string())

In [ ]:
plt.figure(figsize=(9, 4))
combined.groupby("año_lectivo").size().plot(kind="bar", color="#C44E52")
plt.title("Filas por año lectivo")
plt.ylabel("filas")
plt.xlabel("año lectivo")
plt.tight_layout()
plt.show()

### 3b. Análisis bivariado

In [ ]:
# Spearman rather than Pearson: these metrics are strongly skewed and zero-inflated, so a
# Pearson coefficient would be driven by the extreme tail instead of the common range.
correlation = sample_activity.corr(method="spearman")
labels = [column.replace("cantidad_de_", "") for column in correlation.columns]

plt.figure(figsize=(11, 9))
sns.heatmap(
    correlation, annot=True, fmt=".2f", cmap="vlag", center=0,
    vmin=-1, vmax=1, xticklabels=labels, yticklabels=labels,
)
plt.title("Correlación de Spearman entre métricas de actividad")
plt.tight_layout()
plt.show()

In [ ]:
acciones = pd.to_numeric(combined["cantidad_de_acciones_totales"], errors="coerce")
por_año_subsistema = (
    pd.DataFrame(
        {
            "año_lectivo": combined["año_lectivo"],
            "subsistema": combined["subsistema"].astype("object"),
            "acciones": acciones,
        }
    )
    .groupby(["año_lectivo", "subsistema"])["acciones"]
    .median()
    .unstack()
)
display(por_año_subsistema)

por_año_subsistema.plot(marker="o", figsize=(10, 5))
plt.title("Mediana de acciones totales en CREA por año y subsistema")
plt.ylabel("mediana de acciones")
plt.xlabel("año lectivo")
plt.legend(title="subsistema")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
log_acciones = np.log1p(pd.to_numeric(plot_sample["cantidad_de_acciones_totales"], errors="coerce"))
for ax, column in zip(axes.ravel(), ["subsistema", "ciclo", "zona", "contexto"]):
    frame = pd.DataFrame(
        {column: plot_sample[column].astype("object"), "log1p_acciones": log_acciones}
    ).dropna()
    order = frame.groupby(column)["log1p_acciones"].median().sort_values().index
    sns.boxplot(data=frame, x="log1p_acciones", y=column, order=order, ax=ax, fliersize=1)
    ax.set_title(f"Acciones en CREA por {column}", fontsize=10)
    ax.set_ylabel("")
fig.suptitle("Actividad en CREA según variables categóricas (escala log1p)", y=1.0)
fig.tight_layout()
plt.show()

In [ ]:
# Same-year zero activity: the exploratory analogue of the model target. The real target,
# built in section 4, is the *next* year's zero activity for the same student; this view
# only shows which segments are least active today.
zero_now = pd.DataFrame(
    {
        "año_lectivo": combined["año_lectivo"],
        "subsistema": combined["subsistema"].astype("object"),
        "departamento": combined["departamento"].astype("object"),
        "contexto": combined["contexto"].astype("object"),
        "sexo": combined["sexo"].astype("object"),
        "sin_actividad": acciones == 0,
    }
)

por_año_sub = zero_now.pivot_table(
    index="año_lectivo", columns="subsistema", values="sin_actividad", aggfunc="mean"
)
plt.figure(figsize=(11, 5))
sns.heatmap((por_año_sub * 100).round(1), annot=True, fmt=".1f", cmap="Reds")
plt.title("% de estudiantes con cero acciones en CREA, por año y subsistema")
plt.tight_layout()
plt.show()

for column in ["departamento", "contexto", "sexo"]:
    tabla = zero_now.groupby(column)["sin_actividad"].agg(filas="size", sin_actividad="mean")
    tabla["sin_actividad_%"] = (tabla["sin_actividad"] * 100).round(1)
    display(tabla[["filas", "sin_actividad_%"]].sort_values("sin_actividad_%", ascending=False))

In [ ]:
# The missingness is structural, not random: zona, contexto and the Matific metrics exist
# only for primary education, and the PAM columns only for some years. Imputing either
# would fabricate a value that also encodes "which subsystem" or "which year".
estructurales = [
    "zona",
    "contexto",
    "cantidad_de_días_de_ingreso_a_matific",
    "cantidad_de_episodios_finalizados_en_matific",
]
print("Fracción observada de las columnas exclusivas de primaria, por subsistema:")
display(
    combined.groupby(combined["subsistema"].astype("object"))[estructurales]
    .apply(lambda group: group.notna().mean().round(3))
)

pam = ["cantidad_de_días_de_ingreso_a_pam", "cantidad_de_actividades_finalizadas_en_pam"]
print("Fracción observada de las columnas de PAM, por año:")
display(combined.groupby("año_lectivo")[pam].apply(lambda group: group.notna().mean().round(3)))

## 4. Preprocesamiento de datos

Cada regla responde a algo visto en el paso 3:

| Hallazgo del paso 3 | Regla |
| --- | --- |
| `"Sin Dato"` y `"Sin Datos"` conviven como categorías | normalizar texto y mapear los placeholders a `NA` |
| pares `id_persona`+`año_lectivo` duplicados | fusionar por suma si la identidad coincide; descartar el par si no |
| colas extremas en actividad (PAM hasta 57.010) | marcar con una columna `_outlier` sobre el percentil 99,5, sin alterar el valor |
| `grado` con tres notaciones según subsistema | normalizar a dígito + sufijo (`1_p`, `1_c`, `1_t`) |
| faltantes estructurales (primaria, años sin PAM) | **no** imputar acá: el hueco se conserva como `NA` |

Lo que **no** se hace en este paso es imputar. Un año que nunca reportó una métrica tiene
que seguir vacío: rellenarlo genera una constante que después se lee como observación
real. La imputación se ajusta más adelante, y sólo con el conjunto de entrenamiento.

### 4a. Limpieza del dataset combinado

In [ ]:
def resolve_id_year_conflicts(df: pd.DataFrame) -> pd.DataFrame:
    """Merge or discard rows with duplicate id_persona+año_lectivo.

    Rows identical on identity fields (sexo, departamento, etc.) are merged by
    summing activity metrics. Rows with conflicting identity are discarded.
    """
    IDENTITY_COLUMNS = ["sexo", "departamento", "subsistema", "ciclo", "grado", "zona", "contexto"]
    ACTIVITY_COLUMNS = [
        "cantidad_de_días_ingreso_a_crea",
        "cantidad_de_entregas_de_tareas",
        "cantidad_de_comentarios_posteados",
        "cantidad_de_acciones_totales",
        "cantidad_de_días_de_ingreso_a_matific",
        "cantidad_de_episodios_finalizados_en_matific",
        "cantidad_de_días_de_ingreso_a_pam",
        "cantidad_de_actividades_finalizadas_en_pam",
        "cantidad_de_días_de_ingreso_a_biblioteca",
        "cantidad_de_préstamos_en_biblioteca",
    ]

    key = ["id_persona", "año_lectivo"]
    conflict_mask = df.duplicated(subset=key, keep=False)

    if not conflict_mask.any():
        return df

    stable = df[~conflict_mask].copy()
    conflicts = df[conflict_mask]

    merged_rows, dropped = [], 0
    for _, group in conflicts.groupby(key):
        identity_is_consistent = all(
            group[col].dropna().nunique() <= 1 for col in IDENTITY_COLUMNS if col in group.columns
        )
        if identity_is_consistent:
            row = group.iloc[0].copy()
            for col in ACTIVITY_COLUMNS:
                if col in group.columns:
                    row[col] = group[col].sum(min_count=1)
            merged_rows.append(row)
        else:
            dropped += len(group)

    result = pd.concat([stable, pd.DataFrame(merged_rows)], ignore_index=True)
    print(f"id+año conflicts: {len(merged_rows)} pares merged, {dropped} rows discarded (identity mismatch)")
    return result


def add_outlier_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Flag activity metrics above p99.5 percentile."""
    ACTIVITY_COLUMNS = [
        "cantidad_de_comentarios_posteados",
        "cantidad_de_acciones_totales",
        "cantidad_de_actividades_finalizadas_en_pam",
    ]

    for column in ACTIVITY_COLUMNS:
        if column in df.columns:
            threshold = df[column].quantile(0.995)
            df[f"{column}_outlier"] = df[column] > threshold

    return df


# subsistema -> (token to strip from the notation, accepted levels, suffix to append)
GRADO_RULES = {
    "dgeip": ("º", {"1", "2", "3", "4", "5", "6"}, "_p"),
    "dges": ("", {"1", "2", "3", "4", "5", "6", "7", "8", "9"}, "_c"),
    "dgetp": ("ero.", {"0", "1", "2", "3", "4", "7", "8", "9"}, "_t"),
}


def normalize_grado_by_subsistema(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize grado notation by subsistema for consistency.

    The same level is written three ways depending on the subsystem: 1º (primaria),
    1 (ciclo básico), 1ero. (educación media técnica). Each becomes a digit plus a
    subsystem suffix — 1_p / 1_c / 1_t — so one category means one thing. Special
    education categories (sordos, discapacidad, ...) are left literal.

    Applied per subsystem instead of per row: the rules depend on only two columns, and
    ``df.apply(..., axis=1)`` over 4.5M rows is one Python call per row and takes minutes.
    """
    df = df.copy()
    grado = df["grado"].astype("string").str.strip()
    subsistema = df["subsistema"].astype("string").str.strip().str.lower()
    result = grado.copy()

    for name, (strip_token, accepted, suffix) in GRADO_RULES.items():
        in_subsystem = (subsistema == name).fillna(False)
        if not in_subsystem.any():
            continue
        original = grado[in_subsystem]
        stripped = original
        if strip_token:
            stripped = original.str.replace(strip_token, "", regex=False).str.strip()
        # Keep the original text when the level is not one of the accepted ones, so the
        # special-education categories survive untouched.
        result.loc[in_subsystem] = original.where(~stripped.isin(accepted), stripped + suffix)

    df["grado"] = result.astype("object")
    print("\nGRADO normalized by subsistema. Top 25 values post-normalization:")
    print(df["grado"].value_counts().head(25))
    return df


def clean_combined_dataset(df: pd.DataFrame):
    cleaned = df.copy()
    cleaned = cleaned.drop_duplicates()

    # Categories are only a memory device for reading the combined file. The placeholder
    # replacement below tests for the object dtype, and a category column would skip it
    # silently, leaving "Sin Datos" alive as a category.
    category_columns = cleaned.select_dtypes(include="category").columns
    for column in category_columns:
        cleaned[column] = cleaned[column].astype("object")

    for column in cleaned.columns:
        if cleaned[column].dtype == "object":
            cleaned[column] = cleaned[column].astype(str).str.strip().str.lower()
            cleaned[column] = cleaned[column].replace(
                {
                    "": pd.NA,
                    "na": pd.NA,
                    "n/a": pd.NA,
                    "nan": pd.NA,
                    "null": pd.NA,
                    "none": pd.NA,
                    "sin dato": pd.NA,
                    "sin datos": pd.NA,
                    "sin clasificar": pd.NA,
                    "sin_dato": pd.NA,
                    "sin-dato": pd.NA,
                    "unknown": pd.NA,
                    "desconocido": pd.NA,
                }
            )

    if "id_persona" in cleaned.columns:
        cleaned = cleaned.dropna(subset=["id_persona"])

    for column in cleaned.columns:
        if cleaned[column].dtype == "object":
            numeric_values = pd.to_numeric(cleaned[column], errors="coerce")
            valid_ratio = numeric_values.notna().sum() / max(cleaned[column].notna().sum(), 1)
            if valid_ratio > 0.8:
                cleaned[column] = numeric_values

    for column in cleaned.columns:
        if pd.api.types.is_numeric_dtype(cleaned[column]):
            if cleaned[column].dropna().mod(1).eq(0).all():
                cleaned[column] = cleaned[column].astype("Int64")

    cleaned = resolve_id_year_conflicts(cleaned)
    cleaned = add_outlier_flags(cleaned)
    cleaned = normalize_grado_by_subsistema(cleaned)

    # Deliberately no imputation: a year that never reported a measure must stay NA.
    # Filling those gaps invents a constant that later reads as a real observation.
    return cleaned

In [ ]:
def load_or_build_clean(combined_frame):
    """Reuse the cleaned extract when present, otherwise build it and save it.

    Saving is the point of the pipeline: every later section, and
    ``train_engagement_risk.py``, read ``datos_estudiantes_total_clean.csv`` from disk.
    """
    clean_path = DATASETS_DIR / CLEAN_NAME

    if not REBUILD_CLEAN and clean_path.exists():
        print(f"Reusing existing {CLEAN_NAME} (set REBUILD_CLEAN = True to rebuild).")
        return pd.read_csv(clean_path, low_memory=False)

    if combined_frame is None:
        raise ValueError("No combined frame available to clean.")

    started = time.time()
    cleaned_frame = clean_combined_dataset(combined_frame)
    cleaned_frame.to_csv(clean_path, index=False)
    print(f"\nClean dataset saved to: {clean_path}")
    print(f"Cleaned rows: {len(cleaned_frame):,} | columns: {cleaned_frame.shape[1]}")
    print(f"Cleaning took {(time.time() - started) / 60:.1f} min")

    # Read it back so both branches return identical dtypes: the in-memory frame carries
    # pandas nullable Int64 from the cast above, while the training script and every
    # section below work with what read_csv infers.
    return pd.read_csv(clean_path, low_memory=False)


cleaned = load_or_build_clean(combined)
print(f"\nDataset limpio: {len(cleaned):,} filas x {cleaned.shape[1]} columnas")
coverage_clean = report_coverage(cleaned)

In [ ]:
combined_frame = globals().get("combined")
grado_antes = combined_frame["grado"].nunique() if combined_frame is not None else "n/d"

print("Post-condiciones de la limpieza")
print("-" * 64)
print("Pares id_persona+año_lectivo duplicados:", int(cleaned.duplicated(["id_persona", "año_lectivo"]).sum()), "(esperado 0)")
print("Filas duplicadas exactas:               ", int(cleaned.duplicated().sum()), "(esperado 0)")
print("id_persona nulos:                       ", int(cleaned["id_persona"].isna().sum()), "(esperado 0)")

print("\nPlaceholders eliminados — valores de 'zona':")
print(cleaned["zona"].value_counts(dropna=False).to_string())

print(f"\n'grado' normalizado: {cleaned['grado'].nunique()} valores distintos (antes: {grado_antes}).")
print("El total puede subir, y eso es lo buscado: un '1' que antes servía a dos")
print("subsistemas ahora se distingue en 1_c y 1_t.")
print(cleaned["grado"].value_counts().head(25).to_string())

outlier_columns = [column for column in cleaned.columns if column.endswith("_outlier")]
print("\n% de filas marcadas como outlier (umbral p99,5):")
print((cleaned[outlier_columns].mean() * 100).round(3).to_string())

# The gaps have to survive cleaning. A year that never reported PAM must stay empty.
print("\nFaltantes preservados (% por columna, top 8):")
print((cleaned.isna().mean() * 100).round(2).sort_values(ascending=False).head(8).to_string())

In [ ]:
# The combined frame is no longer needed; releasing it keeps peak memory at about one copy.
try:
    del combined, combined_frame, plot_sample, sample_activity, zero_now
except NameError:
    pass
gc.collect()

### 4b. Objetivo, features y partición cronológica

El objetivo se construye un año hacia adelante: las features son las del año *t* y la
etiqueta es "actividad total en CREA igual a cero en *t+1*". Un estudiante entra en la
cohorte sólo si aparece en dos años consecutivos, y si la métrica del año siguiente está
ausente la fila se descarta — un faltante es desconocimiento, no evidencia de cero
actividad.

`choose_activity_column` no se limita a comprobar que la columna exista: exige que en cada
año varíe y alcance el cero. Una métrica constante en un año no puede etiquetarlo y una
sin ceros no produce etiquetas positivas.

In [ ]:
students = normalize_keys(cleaned)
activity_column = choose_activity_column(students)
print("\nMétrica de actividad elegida:", activity_column)

cohort = build_cohort(students, activity_column)
print(f"\nCohorte: {len(cohort):,} pares estudiante-año con el año siguiente observado")
display(
    cohort.groupby("feature_year")["engagement_risk"]
    .agg(filas="size", tasa_de_riesgo="mean")
    .round(4)
)

In [ ]:
model_features = select_model_features(cohort)
print(f"{len(model_features)} features:")
for feature in model_features:
    print("  -", feature)

cohort_years = sorted(int(year) for year in cohort["feature_year"].unique())
train_years, validation_year, test_year = cohort_years[:-2], cohort_years[-2], cohort_years[-1]

train = cohort[cohort["feature_year"].isin(train_years)].copy()
validation = cohort[cohort["feature_year"] == validation_year].copy()
test = cohort[cohort["feature_year"] == test_year].copy()

# Chronological, never random. A random split would let the model learn from 2024 to
# predict 2020. The validation year picks the decision threshold; the test year is only
# ever scored, never fit on and never used to choose anything.
print(f"\nEntrenamiento: años {train_years} -> {len(train):,} filas")
print(f"Validación (elección del umbral): {validation_year} -> {len(validation):,} filas")
print(f"Prueba (evaluación final): {test_year} -> {len(test):,} filas")
for name, frame in {"entrenamiento": train, "validación": validation, "prueba": test}.items():
    print(f"  tasa de riesgo en {name}: {frame['engagement_risk'].mean():.4f}")

In [ ]:
# Preprocessing is fit on the training years only. Fitting the imputer on the whole frame
# would let the test year's medians and category set leak into the training transform.
numeric_features = [column for column in model_features if pd.api.types.is_numeric_dtype(train[column])]
categorical_features = [column for column in model_features if column not in numeric_features]
print("Numéricas  :", numeric_features)
print("Categóricas:", categorical_features)

baseline_pipeline = build_pipeline(train, model_features)
display(baseline_pipeline)

## 5. Definir la métrica de desempeño y construir los modelos

**Métrica principal: PR-AUC** (precisión promedio). Con ~17% de casos positivos, la
exactitud es engañosa —un modelo que predice "sin riesgo" siempre acierta el 83%— y el
ROC-AUC resulta optimista porque premia ordenar bien la clase mayoritaria. El PR-AUC
mide lo que importa: de los estudiantes que el modelo señala, cuántos efectivamente
quedan sin actividad. Su piso es la prevalencia misma, así que el `DummyClassifier`
entra como referencia explícita.

**Métrica secundaria: recall**, con la precisión como restricción. En un programa de
apoyo, no detectar a un estudiante que quedará inactivo cuesta más que contactar a uno
que no lo necesitaba; el umbral se elige maximizando F1 sobre el **año de validación**,
nunca sobre el de prueba.

**Modelos.** Cuatro, de menor a mayor capacidad: prevalencia (piso), regresión logística
balanceada (línea base interpretable), árbol de decisión con profundidad limitada
(no linealidad legible) y gradient boosting (referencia fuerte en datos tabulares).
Los tres primeros comparten exactamente el mismo preprocesamiento, así que una diferencia
de métrica es atribuible al modelo. El boosting usa el suyo, por las razones del
comentario en la celda siguiente.

In [ ]:
def build_boosting_pipeline(train_frame: pd.DataFrame, features: list) -> Pipeline:
    """Gradient boosting with its own preprocessing.

    HistGradientBoosting reads NaN natively and splits on ordinal categories, so imputing,
    scaling and one-hot encoding would only add ~100 sparse columns and throw away the
    "this value was missing" signal that the tree can use directly — which matters here
    because the missingness is structural.
    """
    numeric = [column for column in features if pd.api.types.is_numeric_dtype(train_frame[column])]
    categorical = [column for column in features if column not in numeric]
    preprocessor = ColumnTransformer(
        [
            # Cast to float64 instead of passing through. The numeric block mixes bool,
            # int64 and float64, and ColumnTransformer stacks mixed dtypes into an
            # *object* array — on which this classifier fits 114x slower (68s vs 0.6s on
            # 98k rows) for identical predictions, because every value gets unboxed.
            (
                "numeric",
                FunctionTransformer(
                    np.asarray, kw_args={"dtype": np.float64}, feature_names_out="one-to-one"
                ),
                numeric,
            ),
            (
                "categorical",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=np.nan,
                    encoded_missing_value=np.nan,
                ),
                categorical,
            ),
        ]
    )
    return Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "classifier",
                HistGradientBoostingClassifier(
                    # Column order out of the ColumnTransformer is numeric then categorical.
                    categorical_features=list(range(len(numeric), len(numeric) + len(categorical))),
                    class_weight="balanced",
                    early_stopping=False,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


MODELS = {
    "prevalencia (piso)": lambda frame, features: build_pipeline(
        frame, features, DummyClassifier(strategy="prior", random_state=RANDOM_STATE)
    ),
    "regresión logística": lambda frame, features: build_pipeline(frame, features),
    "árbol de decisión": lambda frame, features: build_pipeline(
        frame,
        features,
        DecisionTreeClassifier(
            max_depth=8, min_samples_leaf=200, class_weight="balanced", random_state=RANDOM_STATE
        ),
    ),
    "gradient boosting": build_boosting_pipeline,
}
print("Modelos a comparar:", list(MODELS))

La validación cruzada es *walk-forward*: el fold *k* entrena con los años anteriores y
valida contra el siguiente. Un K-fold aleatorio mezclaría filas entre años y dejaría a un
modelo entrenado con 2022 validando contra 2020, filtrando información del futuro hacia
un fold "del pasado". El año de prueba no participa de ningún fold.

In [ ]:
dev_years = [int(year) for year in train_years] + [int(validation_year)]
print(f"Validación cruzada walk-forward sobre los años de desarrollo {dev_years}")
print(f"(el año de prueba {test_year} no participa de ningún fold)\n")

cv_by_model = {}
for name, factory in MODELS.items():
    started = time.time()
    folds = time_series_cross_validate(cohort, model_features, dev_years, model_factory=factory)
    cv_by_model[name] = folds
    print(f"{name}:")
    for fold in folds:
        print(
            f"  train={fold['train_years']} val={fold['val_year']}: "
            f"ROC-AUC={fold['roc_auc']:.3f} PR-AUC={fold['pr_auc']:.3f} F1@0.5={fold['f1_at_0.5']:.3f}"
        )
    pr_aucs = [fold["pr_auc"] for fold in folds]
    roc_aucs = [fold["roc_auc"] for fold in folds]
    if pr_aucs:
        print(
            f"  media PR-AUC={np.mean(pr_aucs):.3f} ± {np.std(pr_aucs):.3f} | "
            f"media ROC-AUC={np.mean(roc_aucs):.3f} ± {np.std(roc_aucs):.3f} | "
            f"{time.time() - started:.0f}s\n"
        )

In [ ]:
def evaluate_on_test(name: str, factory) -> tuple:
    """Fit on the training years, pick the threshold on validation, score on test."""
    started = time.time()
    model = factory(train, model_features)
    model.fit(train[model_features], train["engagement_risk"])
    fit_seconds = time.time() - started

    validation_probabilities = model.predict_proba(validation[model_features])[:, 1]
    threshold = float(select_threshold(validation["engagement_risk"], validation_probabilities))

    probabilities = model.predict_proba(test[model_features])[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    y_test = test["engagement_risk"]

    metrics = {
        "modelo": name,
        "umbral": round(threshold, 3),
        "roc_auc": round(float(roc_auc_score(y_test, probabilities)), 4),
        "pr_auc": round(float(average_precision_score(y_test, probabilities)), 4),
        "precision": round(float(precision_score(y_test, predictions, zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, predictions, zero_division=0)), 4),
        "f1": round(float(f1_score(y_test, predictions, zero_division=0)), 4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_test, predictions)), 4),
        "segundos_de_ajuste": round(fit_seconds, 1),
    }
    return model, probabilities, metrics


fitted_models, test_probabilities, rows = {}, {}, []
for name, factory in MODELS.items():
    model, probabilities, metrics = evaluate_on_test(name, factory)
    fitted_models[name] = model
    test_probabilities[name] = probabilities
    rows.append(metrics)
    print(
        f"{name}: PR-AUC={metrics['pr_auc']:.3f} ROC-AUC={metrics['roc_auc']:.3f} "
        f"recall={metrics['recall']:.3f} precision={metrics['precision']:.3f} "
        f"(umbral {metrics['umbral']}, {metrics['segundos_de_ajuste']}s)"
    )

test_results = pd.DataFrame(rows).set_index("modelo")
display(test_results)

## 6. Verificación de supuestos

La regresión logística es la línea base interpretable, así que sus supuestos se verifican
explícitamente. Dos de ellos no se cumplen y conviene decirlo antes de leer cualquier
coeficiente.

### 6a. Multicolinealidad

VIF por feature numérica: mide cuánto se explica cada feature con las demás. Se obtiene
de la diagonal de la matriz de correlación invertida, que es idénticamente el
`1 / (1 - R²)` de la regresión auxiliar de cada feature contra el resto. `statsmodels` no
es dependencia del proyecto y no hace falta: la forma cerrada es más estable que ajustar
una regresión por feature sobre columnas de conteo casi colineales.

In [ ]:
def variance_inflation_factors(frame: pd.DataFrame, columns: list) -> pd.DataFrame:
    """VIF per numeric feature, read off the inverted correlation matrix.

    The i-th diagonal entry of the inverted correlation matrix *is* 1 / (1 - R²) of the
    auxiliary regression of feature i on all the others — the same quantity, without
    fitting one regression per feature. Fitting them explicitly overflows in float64
    here: two of these count columns are close enough to collinear that the least-squares
    solution blows up, which is exactly the condition the diagnostic is meant to report
    rather than crash on.

    The medians fill gaps only to keep this diagnostic's matrix complete; the model's own
    imputation still happens inside its pipeline, fit on training data only.
    """
    matrix = frame[columns].apply(pd.to_numeric, errors="coerce").astype("float64")
    matrix = matrix.fillna(matrix.median())
    keep = [column for column in columns if matrix[column].std() > 0]

    correlation = matrix[keep].corr().to_numpy()
    try:
        inverted = np.linalg.inv(correlation)
    except np.linalg.LinAlgError:
        # Singular only when a feature is an exact linear combination of the others — the
        # extreme case this diagnostic exists to report rather than crash on.
        with np.errstate(all="ignore"):
            inverted = np.linalg.pinv(correlation)
    vif = np.diag(inverted)
    return (
        pd.DataFrame(
            {
                "feature": keep,
                "R2": (1.0 - 1.0 / np.clip(vif, 1e-12, None)).round(4),
                "VIF": vif.round(2),
            }
        )
        .sort_values("VIF", ascending=False)
        .set_index("feature")
    )


vif_sample = train.sample(n=min(200_000, len(train)), random_state=RANDOM_STATE)
vif = variance_inflation_factors(vif_sample, numeric_features)
display(vif)
severe = vif.index[vif["VIF"] > 10].tolist()
print("VIF > 10 (multicolinealidad severa):", severe if severe else "ninguna")
print("Las métricas de CREA miden el mismo comportamiento por vías distintas, así que")
print("cierta colinealidad es esperable: infla el error estándar de los coeficientes")
print("—y por lo tanto su interpretación individual— pero no degrada la predicción.")

### 6b. Linealidad del logit

La regresión logística asume que el log-odds es lineal en cada feature continua. Se
comprueba agrupando por deciles y comparando el log-odds empírico con el valor del
feature.

In [ ]:
def empirical_log_odds(frame: pd.DataFrame, column: str, bins: int = 10) -> pd.DataFrame:
    """Empirical log-odds of the target per quantile bin of one feature."""
    values = pd.to_numeric(frame[column], errors="coerce")
    binned = pd.qcut(values, q=bins, duplicates="drop")
    grouped = frame.groupby(binned, observed=True)["engagement_risk"]
    # Clip so a bin with no positives (or all positives) does not give infinite log-odds.
    rate = grouped.mean().clip(1e-6, 1 - 1e-6)

    result = pd.DataFrame({"log_odds": np.log(rate / (1 - rate)), "n": grouped.size()})
    result["centro_del_bin"] = [interval.mid for interval in result.index]
    return result.reset_index(drop=True)


logit_features = [
    column
    for column in [
        "cantidad_de_acciones_totales",
        "cantidad_de_días_ingreso_a_crea",
        "cantidad_de_entregas_de_tareas",
        "cantidad_de_comentarios_posteados",
    ]
    if column in model_features
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, column in zip(axes.ravel(), logit_features):
    curve = empirical_log_odds(train, column)
    ax.plot(curve["centro_del_bin"], curve["log_odds"], marker="o")
    ax.set_xscale("symlog")
    ax.set_title(column.replace("cantidad_de_", ""), fontsize=9)
    ax.set_xlabel("valor del feature (symlog)")
    ax.set_ylabel("log-odds empírico")
fig.suptitle("Linealidad del logit: log-odds empírico por decil", y=1.0)
fig.tight_layout()
plt.show()

### 6c. Independencia de las observaciones — **supuesto violado**

Cada estudiante aporta hasta seis filas estudiante-año, así que las observaciones no son
independientes. La partición cronológica mantiene honesto el *tiempo*, pero no el
*estudiante*: la fila 2019 de una persona puede estar en entrenamiento mientras su fila
2024 está en prueba. El contraste siguiente acota cuánto del desempeño viene de haber
visto antes a esa misma persona.

In [ ]:
train_students = set(train["id_persona"])
test_students = set(test["id_persona"])
overlap = train_students & test_students

print(f"Estudiantes en entrenamiento: {len(train_students):,}")
print(f"Estudiantes en prueba:        {len(test_students):,}")
print(
    f"Presentes en ambos:           {len(overlap):,} "
    f"({len(overlap) / max(len(test_students), 1) * 100:.1f}% del conjunto de prueba)"
)
print(f"Filas por estudiante en la cohorte: {len(cohort) / cohort['id_persona'].nunique():.2f} en promedio")

unseen = test[~test["id_persona"].isin(train_students)]
print(f"\nFilas de prueba de estudiantes nunca vistos: {len(unseen):,}")
if len(unseen) > 1000 and unseen["engagement_risk"].nunique() == 2:
    logistic = fitted_models["regresión logística"]
    unseen_probabilities = logistic.predict_proba(unseen[model_features])[:, 1]
    print(
        f"PR-AUC sobre todo el año de prueba:      "
        f"{average_precision_score(test['engagement_risk'], test_probabilities['regresión logística']):.4f}"
    )
    print(
        f"PR-AUC sólo en estudiantes no vistos:    "
        f"{average_precision_score(unseen['engagement_risk'], unseen_probabilities):.4f}"
    )
    print("Una caída grande indicaría que el modelo memoriza estudiantes en lugar de")
    print("aprender el patrón; una diferencia chica es la evidencia de que generaliza.")
else:
    print("Muy pocos estudiantes no vistos para una comparación estable.")

### 6d. Balance de clases y eventos por variable

In [ ]:
display(
    cohort.groupby("feature_year")["engagement_risk"]
    .agg(filas="size", positivos="sum", tasa="mean")
    .round(4)
)

preprocessor = fitted_models["regresión logística"].named_steps["preprocessor"]
encoded_columns = preprocessor.transform(train[model_features].head(1000)).shape[1]
positives = int(train["engagement_risk"].sum())
print(f"Columnas después de codificar: {encoded_columns}")
print(f"Eventos positivos en entrenamiento: {positives:,}")
print(f"Eventos por columna: {positives / max(encoded_columns, 1):,.0f} (la regla usual pide >= 10)")
print("\nEl desbalance (~17% de positivos) se trata con class_weight='balanced' y")
print("midiendo con PR-AUC, no con exactitud.")

### 6e. Influencia de los valores extremos

Los outliers marcados en el paso 4 se conservan en el dataset. Acá se mide si el ajuste
depende de ellos: se reentrena la línea base sin esas filas y se compara.

In [ ]:
outlier_features = [column for column in model_features if column.endswith("_outlier")]
flagged = np.zeros(len(train), dtype=bool)
for column in outlier_features:
    flagged |= train[column].astype("boolean").fillna(False).to_numpy(dtype=bool)

print(f"Filas marcadas como outlier en entrenamiento: {flagged.sum():,} ({flagged.mean() * 100:.2f}%)")

trimmed = train.loc[~flagged]
trimmed_model = build_pipeline(trimmed, model_features)
trimmed_model.fit(trimmed[model_features], trimmed["engagement_risk"])
trimmed_pr = average_precision_score(
    test["engagement_risk"], trimmed_model.predict_proba(test[model_features])[:, 1]
)
full_pr = average_precision_score(test["engagement_risk"], test_probabilities["regresión logística"])

print(f"PR-AUC con todas las filas:     {full_pr:.4f}")
print(f"PR-AUC excluyendo los outliers: {trimmed_pr:.4f}")
print(f"Diferencia: {trimmed_pr - full_pr:+.4f} — si es despreciable, los extremos no dominan el ajuste")

### 6f. Calibración

El umbral se elige sobre probabilidades, así que importa qué tan literales son. Con
`class_weight="balanced"` la regresión logística sobreestima la probabilidad de la clase
positiva por construcción: la curva quedará por debajo de la diagonal. No invalida el
ranking —ni el PR-AUC, que sólo depende del orden— pero significa que un 0,7 predicho no
es un 70% de probabilidad real.

In [ ]:
plt.figure(figsize=(7, 6))
for name in [n for n in ["regresión logística", "árbol de decisión", "gradient boosting"] if n in test_probabilities]:
    observed, predicted = calibration_curve(
        test["engagement_risk"], test_probabilities[name], n_bins=20, strategy="quantile"
    )
    plt.plot(predicted, observed, marker="o", label=name)
plt.plot([0, 1], [0, 1], "k--", label="calibración perfecta")
plt.xlabel("probabilidad predicha media")
plt.ylabel("proporción observada de positivos")
plt.title(f"Curvas de calibración — año de prueba {test_year}")
plt.legend()
plt.tight_layout()
plt.show()

### 6g. Ausencia de fuga temporal y faltantes estructurales

In [ ]:
assert max(train_years) < validation_year < test_year, "La partición no es cronológica"
assert "next_year_crea_activity" not in model_features, "El objetivo entró como feature"
assert "engagement_risk" not in model_features, "El objetivo entró como feature"
assert not set(train["feature_year"]) & {validation_year, test_year}, "Años solapados entre particiones"
print("Sin fuga temporal:", train_years, "<", validation_year, "<", test_year)
print("El objetivo y toda columna derivada del año etiqueta están fuera de las features.")

structural = [column for column in ["zona", "contexto"] if column in model_features]
print("\nFaltantes estructurales entre las features:", structural)
print("Existen sólo para subsistema == 'dgeip'. El imputador las completa y su indicador")
print("de faltante funciona como proxy de 'no es primaria', así que esos coeficientes")
print("quedan confundidos con el subsistema y no deben leerse como efecto de la zona.")

## 7. Comparar modelos y determinar el mejor

In [ ]:
comparison_rows = []
for name in MODELS:
    folds = cv_by_model[name]
    pr_aucs = [fold["pr_auc"] for fold in folds]
    roc_aucs = [fold["roc_auc"] for fold in folds]
    row = {
        "modelo": name,
        "cv_pr_auc_media": round(float(np.mean(pr_aucs)), 4) if pr_aucs else None,
        "cv_pr_auc_desv": round(float(np.std(pr_aucs)), 4) if pr_aucs else None,
        "cv_roc_auc_media": round(float(np.mean(roc_aucs)), 4) if roc_aucs else None,
    }
    row.update(test_results.loc[name].to_dict())
    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows).set_index("modelo").sort_values("pr_auc", ascending=False)
display(comparison)

baseline_pr = float(test_results.loc["prevalencia (piso)", "pr_auc"])
print(f"Piso de referencia (prevalencia): PR-AUC = {baseline_pr:.4f}")
print(f"Tasa de riesgo real en {test_year}: {test['engagement_risk'].mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
y_test = test["engagement_risk"]

for name in MODELS:
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_probabilities[name])
    axes[0].plot(recall_curve, precision_curve, label=name)
    false_positive_rate, true_positive_rate, _ = roc_curve(y_test, test_probabilities[name])
    axes[1].plot(false_positive_rate, true_positive_rate, label=name)

axes[0].axhline(y_test.mean(), color="k", ls="--", label="prevalencia")
axes[0].set_xlabel("recall")
axes[0].set_ylabel("precisión")
axes[0].set_title("Curva Precision-Recall")
axes[0].legend(fontsize=8)

axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_xlabel("tasa de falsos positivos")
axes[1].set_ylabel("tasa de verdaderos positivos")
axes[1].set_title("Curva ROC")
axes[1].legend(fontsize=8)

ordered = comparison.sort_values("pr_auc")
axes[2].barh(list(ordered.index), ordered["pr_auc"].to_numpy(), color="#4C72B0")
axes[2].axvline(baseline_pr, color="k", ls="--", label="piso de prevalencia")
axes[2].set_xlabel("PR-AUC en el año de prueba")
axes[2].set_title("PR-AUC por modelo")
axes[2].legend(fontsize=8)

fig.suptitle(f"Comparación de modelos — año de prueba {test_year}", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# The development folds decide, not the test year: picking the winner by its test score
# would turn the held-out year into a selection set and inflate the number reported for it.
ranked = comparison.sort_values("cv_pr_auc_media", ascending=False)
best_model_name = str(ranked.index[0])

print("Mejor modelo por PR-AUC en validación cruzada:", best_model_name)
print(f"  CV PR-AUC   = {ranked.loc[best_model_name, 'cv_pr_auc_media']:.4f} ± {ranked.loc[best_model_name, 'cv_pr_auc_desv']:.4f}")
print(f"  Test PR-AUC = {ranked.loc[best_model_name, 'pr_auc']:.4f} (piso {baseline_pr:.4f})")
print(f"  Test recall = {ranked.loc[best_model_name, 'recall']:.4f} | precisión = {ranked.loc[best_model_name, 'precision']:.4f}")

ARTIFACTS_DIR.mkdir(exist_ok=True)
comparison.to_csv(ARTIFACTS_DIR / "model_comparison.csv")
(ARTIFACTS_DIR / "model_comparison.json").write_text(
    json.dumps(
        {
            "activity_column": activity_column,
            "train_years": [int(year) for year in train_years],
            "validation_year": int(validation_year),
            "test_year": int(test_year),
            "test_rows": int(len(test)),
            "test_risk_rate": float(test["engagement_risk"].mean()),
            "best_model": best_model_name,
            "models": json.loads(comparison.reset_index().to_json(orient="records")),
            "cross_validation": cv_by_model,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
dump(fitted_models[best_model_name], ARTIFACTS_DIR / "best_model_pipeline.joblib")
print("\nGuardado en artifacts/: model_comparison.csv, model_comparison.json, best_model_pipeline.joblib")

## 8. Observaciones e insights de negocio

### 8a. Qué mueve la predicción

In [ ]:
logistic = fitted_models["regresión logística"]
encoded_names = logistic.named_steps["preprocessor"].get_feature_names_out()
coefficients = pd.Series(logistic.named_steps["classifier"].coef_[0], index=encoded_names)
top = pd.concat([coefficients.nlargest(12), coefficients.nsmallest(12)]).sort_values()

plt.figure(figsize=(10, 8))
colors = ["#C44E52" if value > 0 else "#4C72B0" for value in top.to_numpy()]
plt.barh([name.split("__")[-1] for name in top.index], top.to_numpy(), color=colors)
plt.axvline(0, color="k", lw=0.8)
plt.title("Regresión logística: coeficientes de mayor peso\n(rojo = aumenta el riesgo, azul = lo reduce)")
plt.xlabel("coeficiente (features numéricas estandarizadas)")
plt.tight_layout()
plt.show()

In [ ]:
# Permutation importance on the winning model: how much PR-AUC is lost when one feature is
# shuffled. Unlike a coefficient it needs no linearity assumption, so it is comparable
# across the four models. Computed on a sample because it refits nothing but predicts
# once per feature per repeat.
importance_sample = test.sample(n=min(20_000, len(test)), random_state=RANDOM_STATE)
importance = permutation_importance(
    fitted_models[best_model_name],
    importance_sample[model_features],
    importance_sample["engagement_risk"],
    n_repeats=3,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=1,
)
ranking = (
    pd.DataFrame(
        {
            "feature": model_features,
            "caída_de_PR_AUC": importance.importances_mean.round(5),
            "desv": importance.importances_std.round(5),
        }
    )
    .sort_values("caída_de_PR_AUC", ascending=False)
    .set_index("feature")
)
display(ranking.head(15))

### 8b. Dónde se concentra el riesgo

In [ ]:
for column in ["subsistema", "ciclo", "grado", "departamento", "contexto", "zona", "sexo"]:
    if column not in test.columns:
        continue
    tabla = test.groupby(test[column].astype("object"), dropna=False)["engagement_risk"].agg(
        estudiantes="size", riesgo="mean"
    )
    tabla["riesgo_%"] = (tabla["riesgo"] * 100).round(1)
    display(
        tabla[["estudiantes", "riesgo_%"]]
        .sort_values("riesgo_%", ascending=False)
        .head(12)
        .rename_axis(column)
    )

### 8c. Traducción operativa: a quién contactar

La pregunta de gestión no es "qué tan bueno es el clasificador" sino "si el programa de
apoyo puede llegar al N% de los estudiantes, qué proporción de los que van a quedar
inactivos está en ese N%". Eso es lo que mide la tabla de deciles.

In [ ]:
probabilities = test_probabilities[best_model_name]
# Rank before cutting: the boosting model ties many students at the same probability, and
# qcut on tied values collapses the deciles.
deciles = pd.qcut(pd.Series(probabilities).rank(method="first"), 10, labels=False)
lift = pd.DataFrame(
    {"decil": 9 - deciles.to_numpy(), "riesgo": test["engagement_risk"].to_numpy()}
)

tabla = lift.groupby("decil")["riesgo"].agg(estudiantes="size", casos="sum", tasa="mean")
tabla["riesgo_%"] = (tabla["tasa"] * 100).round(1)
tabla["captura_acumulada_%"] = (tabla["casos"].cumsum() / tabla["casos"].sum() * 100).round(1)
tabla["lift"] = (tabla["tasa"] / test["engagement_risk"].mean()).round(2)
display(tabla[["estudiantes", "casos", "riesgo_%", "captura_acumulada_%", "lift"]])

decil_0 = float(tabla["captura_acumulada_%"].iloc[0])
decil_1 = float(tabla["captura_acumulada_%"].iloc[1])
print(f"Contactando al 10% de mayor riesgo predicho se alcanza el {decil_0:.1f}% de los casos.")
print(f"Contactando al 20% de mayor riesgo predicho se alcanza el {decil_1:.1f}% de los casos.")

### 8d. Observaciones

**Sobre los datos**

1. **Todas las métricas de actividad tienen exceso de ceros y cola larga.** La mediana de
   varias es 0 y el máximo llega a decenas de miles. Cualquier lectura en escala lineal
   engaña; los gráficos del paso 3 usan `log1p` por eso.
2. **Los faltantes no son aleatorios y no deben imputarse en la limpieza.** `zona`,
   `contexto` y las métricas de Matific existen en el 100% de las filas de primaria
   (`dgeip`) y en el 0% del resto: es un patrón estructural, no un hueco. Las columnas de
   PAM faltan por completo en 2023 y 2025, así que no son utilizables para las cohortes
   recientes.
3. **El extracto 2025 renombra tres métricas de CREA** con el sufijo `en CREA`.
   `COLUMN_ALIASES` las unifica; sin eso cada nombre queda vacío en los años que usan la
   otra variante, y ese vacío se lee después como dato real.
4. **`rol` es constante** (100% `estudiante`): no aporta señal y `select_model_features`
   lo descarta por medición, no por nombre.
5. **La actividad no es estable entre años.** La mediana de acciones por año y subsistema
   (paso 3b) sube marcadamente en 2020 —el año de la enseñanza remota— y vuelve a bajar
   después. Un modelo entrenado sobre años previos y evaluado en un año posterior tiene
   que soportar ese cambio de nivel; es exactamente lo que mide la validación
   walk-forward.

**Sobre el modelo**

6. **El desempeño está muy por encima del piso, y es estable.** El PR-AUC del mejor
   modelo contra el ~0,17 de la prevalencia (tabla del paso 7) indica señal real; la
   desviación entre folds dice cuánto de esa ventaja es robusta al año.
7. **La ganancia por capacidad del modelo es menor que la ganancia por datos limpios.**
   Comparar las cuatro filas del paso 7: la distancia entre la regresión logística y el
   boosting es chica frente a la distancia de cualquiera de ellos al piso. Con datos
   tabulares y features de conteo, elegir bien el objetivo y la partición pesa más que
   elegir el algoritmo.
8. **La actividad del año en curso es el predictor dominante.** Coeficientes e importancia
   por permutación (paso 8a) apuntan a las métricas de CREA del propio año: quien está
   poco activo hoy es quien más probablemente quede inactivo el año próximo.
9. **Dos supuestos no se cumplen y limitan la lectura.** Las observaciones no son
   independientes (el mismo estudiante aporta varias filas) y hay colinealidad entre las
   métricas de CREA. Ninguna de las dos arruina la predicción, pero ambas impiden leer un
   coeficiente individual como efecto causal.

**Recomendaciones**

10. **Usarlo para priorizar alcance, no para decidir sobre personas.** La tabla de deciles
    del paso 8c es la forma correcta de usar el modelo: ordenar a quién contactar primero
    con una capacidad limitada. El modelo no explica *por qué* un estudiante se desconecta.
11. **Revisar la granularidad de PAM con el equipo responsable.** Se detectaron 179
    registros con más de 10.000 actividades finalizadas (media de 457 por día de ingreso),
    lo que sugiere que PAM cuenta ejercicios o intentos individuales. Las banderas
    `_outlier` los aíslan, pero la definición de la métrica debería confirmarse.
12. **Reconciliar los nombres de columna en el origen.** El renombrado de 2025 y la
    ausencia de PAM en 2023 y 2025 son la causa de la mayor parte de la complejidad de la
    limpieza. Un esquema estable entre años elimina `COLUMN_ALIASES` y hace comparables
    las cohortes recientes.
13. **Reevaluar cada año.** El cambio de nivel de 2020 muestra que la relación entre
    features y objetivo se mueve con el contexto. La validación walk-forward ya está
    escrita para reejecutarse agregando el año nuevo.

**Límite de uso.** Este modelo estima una probabilidad de baja actividad en una
plataforma; no mide aprendizaje, ni esfuerzo, ni riesgo de desvinculación educativa. Su
uso previsto es dimensionar y priorizar apoyo. No debe utilizarse para decisiones
automatizadas sobre estudiantes individuales, ni para clasificar o etiquetar personas.